# Session 7.1: CNN Fundamentals - MNIST Digit Classification

**Learning Objectives:**
- Understand Convolutional Neural Networks (CNNs)
- Build a CNN from scratch for image classification
- Achieve >98% accuracy on MNIST dataset
- Visualize CNN filters and feature maps
- Compare CNN vs Dense Neural Network performance

**Time Estimate:** 90-120 minutes

---

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Sklearn utilities
from sklearn.metrics import confusion_matrix, classification_report

# Display settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Load and Explore MNIST Dataset

MNIST is a classic dataset of handwritten digits (0-9):
- 60,000 training images
- 10,000 test images
- Each image is 28x28 pixels, grayscale

In [ ]:
# Load MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"\nPixel value range: {X_train.min()} to {X_train.max()}")
print(f"Unique classes: {np.unique(y_train)}")

### Visualization 1: Sample MNIST Digits

In [ ]:
# Display sample images
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('Sample MNIST Digits', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

### Visualization 2: Class Distribution

In [ ]:
# Check class distribution
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(10, 6))
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.title('Distribution of Digits in Training Set', fontsize=14, fontweight='bold')
plt.xlabel('Digit', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(unique)
plt.grid(axis='y', alpha=0.3)

for i, count in enumerate(counts):
    plt.text(i, count + 100, str(count), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("Dataset is well-balanced across all digit classes.")

## 3. Data Preprocessing

Steps:
1. Reshape images to include channel dimension (28, 28, 1)
2. Normalize pixel values to [0, 1]
3. One-hot encode labels

In [ ]:
# Reshape to add channel dimension
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# Normalize pixel values to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Preprocessed training shape: {X_train.shape}")
print(f"Preprocessed test shape: {X_test.shape}")
print(f"One-hot encoded labels shape: {y_train_cat.shape}")
print(f"Example one-hot encoding for digit {y_train[0]}: {y_train_cat[0]}")

## 4. Build CNN Architecture

**CNN Architecture:**
- Conv2D(32 filters, 3x3) → ReLU → MaxPooling2D(2x2)
- Conv2D(64 filters, 3x3) → ReLU → MaxPooling2D(2x2)
- Conv2D(64 filters, 3x3) → ReLU
- Flatten
- Dense(64) → ReLU → Dropout(0.5)
- Dense(10) → Softmax

### LLM Prompt for Code Generation:
```
Create a Keras Sequential CNN model for MNIST digit classification with:
- Input shape: (28, 28, 1)
- First conv block: 32 filters, 3x3, ReLU, 2x2 max pooling
- Second conv block: 64 filters, 3x3, ReLU, 2x2 max pooling  
- Third conv layer: 64 filters, 3x3, ReLU
- Flatten layer
- Dense layer with 64 units, ReLU, 50% dropout
- Output layer with 10 units and softmax activation
```

In [ ]:
def create_cnn_model():
    """
    Build a Convolutional Neural Network for MNIST classification.
    
    Returns:
        model: Compiled Keras model
    """
    model = models.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Third Convolutional Layer
        layers.Conv2D(64, (3, 3), activation='relu'),
        
        # Flatten and Dense Layers
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        
        # Output Layer
        layers.Dense(10, activation='softmax')
    ])
    
    return model

# Create model
cnn_model = create_cnn_model()

# Display model architecture
cnn_model.summary()

### Visualization 3: Model Architecture Diagram

In [ ]:
# Visualize model architecture
keras.utils.plot_model(
    cnn_model,
    show_shapes=True,
    show_layer_names=True,
    rankdir='TB',
    expand_nested=True,
    dpi=96
)

## 5. Compile and Train the Model

In [ ]:
# Compile model
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully.")
print(f"Optimizer: Adam")
print(f"Loss function: Categorical Crossentropy")
print(f"Metrics: Accuracy")

In [ ]:
# Train model
print("Training CNN model...\n")

history = cnn_model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

print("\nTraining completed!")

### Visualization 4: Training History

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
ax1.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Loss plot
ax2.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")

## 6. Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = cnn_model.evaluate(X_test, y_test_cat, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

if test_accuracy > 0.98:
    print("\n✓ SUCCESS: Achieved >98% accuracy target!")
else:
    print(f"\n⚠ Test accuracy is {test_accuracy:.2%}. Target is >98%.")

### Visualization 5: Confusion Matrix

In [ ]:
# Make predictions
y_pred = cnn_model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix - CNN on MNIST', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classes, digits=4))

### Visualization 6: Sample Predictions

In [ ]:
# Display sample predictions
num_samples = 16
random_indices = np.random.choice(len(X_test), num_samples, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Sample Predictions', fontsize=16, fontweight='bold')

for i, (ax, idx) in enumerate(zip(axes.flat, random_indices)):
    # Display image
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    
    # Get prediction and confidence
    pred = y_pred_classes[idx]
    true = y_test[idx]
    confidence = y_pred[idx][pred]
    
    # Color code: green for correct, red for incorrect
    color = 'green' if pred == true else 'red'
    
    ax.set_title(f'True: {true}, Pred: {pred}\nConf: {confidence:.2%}',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

### Visualization 7: Error Analysis

In [ ]:
# Find misclassified examples
misclassified_indices = np.where(y_pred_classes != y_test)[0]

print(f"Total misclassified samples: {len(misclassified_indices)}")
print(f"Error rate: {len(misclassified_indices)/len(y_test):.2%}")

if len(misclassified_indices) > 0:
    # Display first 12 misclassified examples
    num_errors = min(12, len(misclassified_indices))
    
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    fig.suptitle('Misclassified Examples', fontsize=16, fontweight='bold')
    
    for i, ax in enumerate(axes.flat[:num_errors]):
        idx = misclassified_indices[i]
        
        ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
        
        pred = y_pred_classes[idx]
        true = y_test[idx]
        confidence = y_pred[idx][pred]
        
        ax.set_title(f'True: {true}, Pred: {pred}\nConf: {confidence:.2%}',
                     fontsize=9, color='red', fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("\nPerfect classification! No errors found.")

## 7. Comparison: CNN vs Dense Network

Let's build a simple dense network to compare performance.

In [ ]:
# Build dense network
dense_model = models.Sequential([
    layers.Flatten(input_shape=(28, 28, 1)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

dense_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Dense Network Architecture:")
dense_model.summary()

# Train dense network
print("\nTraining Dense Network...")
dense_history = dense_model.fit(
    X_train, y_train_cat,
    batch_size=128,
    epochs=10,
    validation_split=0.2,
    verbose=0
)

# Evaluate
dense_test_loss, dense_test_accuracy = dense_model.evaluate(X_test, y_test_cat, verbose=0)

print(f"\nDense Network Test Accuracy: {dense_test_accuracy:.4f}")
print(f"CNN Test Accuracy: {test_accuracy:.4f}")
print(f"\nCNN Improvement: {(test_accuracy - dense_test_accuracy):.4f} ({((test_accuracy - dense_test_accuracy)/dense_test_accuracy)*100:.2f}%)")

## 8. Save the Model

In [ ]:
# Save model
model_path = 'mnist_cnn_model.h5'
cnn_model.save(model_path)
print(f"Model saved to {model_path}")

# Save model in SavedModel format (for deployment)
cnn_model.save('mnist_cnn_savedmodel')
print("Model also saved in SavedModel format for deployment.")

## 9. Key Takeaways

### What We Learned:
1. **CNN Architecture**: Convolutional layers extract spatial features from images
2. **Max Pooling**: Reduces spatial dimensions while retaining important features
3. **Performance**: CNNs significantly outperform dense networks for image tasks
4. **MNIST**: Achieved >98% accuracy on handwritten digit classification

### Why CNNs Work Better:
- **Local Connectivity**: Each neuron only looks at a small region
- **Parameter Sharing**: Same filters applied across entire image
- **Translation Invariance**: Can detect features regardless of position
- **Hierarchical Features**: Early layers detect edges, later layers detect complex patterns

### Next Steps:
- Session 7.2: Build deeper CNNs for CIFAR-10 (color images, more complex)
- Session 7.3: Transfer learning with pre-trained models
- Session 7.4: Object detection with YOLO

---

## 10. Practice Exercises

### Exercise 1: Experiment with Architecture
Try modifying the CNN architecture:
- Add more convolutional layers
- Change filter sizes (5x5 instead of 3x3)
- Add batch normalization layers
- Compare performance

### Exercise 2: Data Augmentation
Add data augmentation to improve generalization:
- Rotation (±10 degrees)
- Width/height shift (±10%)
- Zoom (±10%)

### Exercise 3: Fashion MNIST
Apply this same CNN to Fashion MNIST dataset:
```python
from tensorflow.keras.datasets import fashion_mnist
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
```

### Troubleshooting Tips:
- **Low accuracy**: Increase epochs, add more layers, or use data augmentation
- **Overfitting**: Increase dropout rate, add more regularization
- **Slow training**: Reduce batch size, use GPU if available
- **Memory errors**: Reduce batch size or model complexity

---

**Session Complete!** You've successfully built a CNN that achieves >98% accuracy on MNIST.
